# Training the GST Regulatory LLM — Kaggle / Colab

Trains the 131.5M param model from this project's series on the extracted
GST regulatory corpus. Works on either Kaggle or Colab — Drive-mounting
for Colab checkpoint persistence is handled below since **Colab's local
disk does NOT survive a session disconnect** — losing checkpoints to that
is the single most common way to lose hours of training progress here.

Before running: set `HF_DATASET_REPO` in the config cell to your actual
Hugging Face dataset repo id (e.g. `your-username/gst-rulings-corpus`).


In [ ]:
!pip install -q torch numpy tiktoken datasets

In [ ]:
# ---- Config — EDIT THIS ----
MODEL_DIR = "../../04_model-architecture/01_main-chapter-code" # Local/Jupyter path
# MODEL_DIR = "/content/fingpt-131m-project/04_model-architecture/01_main-chapter-code" # Colab path
# MODEL_DIR = "/kaggle/working/fingpt-131m-project/04_model-architecture/01_main-chapter-code" # Kaggle path

HF_DATASET_REPO = "your-username/gst-rulings-corpus"  # <-- your real HF repo id
CONTEXT_LEN = 1024
BATCH_SIZE = 8
MAX_STEPS = 20000
EVAL_EVERY = 250
EVAL_ITERS = 50
LR = 3e-4
WEIGHT_DECAY = 0.1
PATIENCE = 10  # early-stopping: rounds with no val improvement before stopping


## Persistent checkpoint storage

**Colab:** mounts Google Drive — checkpoints go there, not local disk,
so a disconnected/recycled runtime doesn't lose your progress.

**Kaggle:** uses `/kaggle/working/` — persists for the session and is
saved when you commit the notebook, but isn't the same as Drive's
cross-session persistence. Commit periodically on long runs, don't rely
solely on the working directory surviving an unexpected kernel restart.


In [ ]:
import os

IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/llm-from-scratch/checkpoints"
    DATA_CACHE_DIR = "/content/drive/MyDrive/llm-from-scratch/data"
else:
    # Assume Kaggle or another environment with a working directory
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    DATA_CACHE_DIR = "/kaggle/working/data"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATA_CACHE_DIR, exist_ok=True)
print(f"Checkpoints -> {CHECKPOINT_DIR}")
print(f"Data cache  -> {DATA_CACHE_DIR}")


## Load corpus from the published HF dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset(HF_DATASET_REPO)
print(ds)
print()
print("Sample:", ds["train"][0]["text"][:200])


## Tokenize + pack

Same logic as `tokenize_and_pack.py` in the project repo — concatenates
documents with an EOT separator, slices into fixed-length sequences, no
padding. Runs in-memory here rather than writing .bin files, so it's
self-contained in the notebook.


In [ ]:
import numpy as np
import tiktoken

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token

def tokenize_and_pack_split(hf_split, context_len):
    all_ids = []
    for row in hf_split:
        all_ids.extend(enc.encode(row["text"]))
        all_ids.append(EOT)
    arr = np.array(all_ids, dtype=np.uint16)
    n_seq = len(arr) // context_len
    return arr[: n_seq * context_len].reshape(n_seq, context_len)

train_data = tokenize_and_pack_split(ds["train"], CONTEXT_LEN)
val_data = tokenize_and_pack_split(ds["validation"], CONTEXT_LEN)
print(f"Train sequences: {train_data.shape[0]:,}")
print(f"Val sequences:   {val_data.shape[0]:,}")


## Model architecture (from `model.py`)

In [ ]:
import sys
sys.path.insert(0, MODEL_DIR)
from model import GPTConfig, GPTModel


In [ ]:
# Sanity check before committing to a real run
cfg = GPTConfig(context_length=CONTEXT_LEN)
model = GPTModel(cfg)
print(f"Total parameters: {model.num_params():,}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
model = model.to(device)


## Training loop (from `train.py`, adapted for notebook cells)

In [ ]:
def get_batch(data, batch_size, device):
    idx = np.random.randint(0, data.shape[0], size=batch_size)
    seqs = torch.from_numpy(data[idx].astype(np.int64))
    x = seqs[:, :-1].contiguous()
    y = seqs[:, 1:].contiguous()
    return x.to(device), y.to(device)


def configure_optimizer(model, weight_decay, lr):
    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (decay_params if param.dim() >= 2 else no_decay_params).append(param)
    return torch.optim.AdamW([
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=lr, betas=(0.9, 0.95))


@torch.no_grad()
def estimate_loss(model, data, batch_size, device, eval_iters=50):
    model.eval()
    losses = torch.zeros(eval_iters)
    for i in range(eval_iters):
        x, y = get_batch(data, batch_size, device)
        _, loss = model(x, y)
        losses[i] = loss.item()
    model.train()
    return losses.mean().item()


def save_checkpoint(path, model, optimizer, step, best_val_loss, cfg):
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "best_val_loss": best_val_loss,
        "config": vars(cfg),
    }, path)


In [ ]:
# Resume support — if a checkpoint already exists in CHECKPOINT_DIR (e.g.
# from a previous session that got cut off), pick up from there instead
# of starting over. Critical for Colab's disconnect-prone free tier.
import os as _os

last_ckpt_path = _os.path.join(CHECKPOINT_DIR, "last.pt")
optimizer = configure_optimizer(model, WEIGHT_DECAY, LR)

start_step = 0
best_val_loss = float("inf")
no_improve_count = 0

if _os.path.exists(last_ckpt_path):
    print(f"Found existing checkpoint at {last_ckpt_path} — resuming.")
    ckpt = torch.load(last_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_step = ckpt["step"]
    best_val_loss = ckpt["best_val_loss"]
    print(f"  Resumed at step {start_step}, best_val_loss so far: {best_val_loss:.4f}")
else:
    print("No existing checkpoint — starting fresh.")


In [ ]:
import time

model.train()
t0 = time.time()

for step in range(start_step, MAX_STEPS):
    x, y = get_batch(train_data, BATCH_SIZE, device)
    _, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step % EVAL_EVERY == 0 or step == MAX_STEPS - 1:
        train_loss_est = estimate_loss(model, train_data, BATCH_SIZE, device, EVAL_ITERS)
        val_loss = estimate_loss(model, val_data, BATCH_SIZE, device, EVAL_ITERS)
        elapsed = time.time() - t0
        print(f"step {step:6d} | train_loss {train_loss_est:.4f} | "
              f"val_loss {val_loss:.4f} | {elapsed:.0f}s elapsed")

        save_checkpoint(last_ckpt_path, model, optimizer, step, best_val_loss, cfg)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve_count = 0
            save_checkpoint(_os.path.join(CHECKPOINT_DIR, "best.pt"),
                             model, optimizer, step, best_val_loss, cfg)
            print(f"  New best val_loss: {best_val_loss:.4f} — saved best.pt")
        else:
            no_improve_count += 1
            print(f"  No improvement ({no_improve_count}/{PATIENCE})")

        if no_improve_count >= PATIENCE:
            print(f"\nEarly stopping at step {step} — best.pt is your model, not last.pt.")
            break

print("\nTraining complete (or interrupted — re-run this cell to resume from last.pt).")
